# Project Echo - Train and Save EfficientNetV2 Model

**Goal:** reload the audio dataset, extract Mel spectrogram features, train the selected EfficientNetV2 model, evaluate it, and save the model files needed for the Engine pipeline.

This notebook produces:

- `efficientnetv2_project_echo.pt`
- `class_mapping.json`
- `preprocess_config.json`
- `training_metrics.json`
- optional feature cache under `_feature_cache`

Run the cells in order.


## 1. Install required packages

Run this once if your environment does not already have these packages.

`librosa==0.9.2` is used because newer versions previously caused kernel issues in the Project Echo benchmark work.


In [ ]:
# Run this only if needed.
# If packages are already installed, you can skip this cell.

!pip install librosa==0.9.2 timm scikit-learn tqdm soundfile


## 2. Imports


In [7]:
import os
import json
import time
import pathlib
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import librosa

from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

import timm


## 3. Configuration

Change `DATA_ROOT` to your local Project Echo dataset path.

Expected structure:

```text
data_files/
    class_or_species_name_1/
        audio1.wav
    class_or_species_name_2/
        audio2.wav
```

If your dataset has another structure, this notebook still scans all subfolders and uses the **parent folder name** as the label.


In [12]:
# EDIT THIS PATH IF NEEDED
DATA_ROOT = r"C:\Deakin\ProjectEcho\Project-Echo\src\Prototypes\data\data_files"

VALID_EXTS = {".wav", ".mp3", ".flac", ".ogg"}

# Same feature settings used in previous benchmark work
TARGET_SR = 32000
DURATION_S = 5.0
N_MELS = 128
HOP_LENGTH = 512
FMIN = 20
FMAX = 14000

TEST_SIZE = 0.2
RANDOM_STATE = 42

# Training settings
EPOCHS = 3          # Start with 3 to confirm it works. Increase later, e.g. 10.
BATCH_SIZE = 16
LEARNING_RATE = 1e-4

DATA_ROOT = pathlib.Path(DATA_ROOT)

CACHE_DIR = DATA_ROOT / "_feature_cache"
CACHE_DIR.mkdir(exist_ok=True)

MODEL_DIR = pathlib.Path(
    r"C:\Deakin\ProjectEcho\Project-Echo\src\Prototypes\engine\torch_impl\Integrate_EfficientNetV2_Engine\_trained_models"
)
MODEL_DIR.mkdir(exist_ok=True)

print("Data root:", DATA_ROOT)
print("Feature cache:", CACHE_DIR)
print("Model output folder:", MODEL_DIR)


Data root: C:\Deakin\ProjectEcho\Project-Echo\src\Prototypes\data\data_files
Feature cache: C:\Deakin\ProjectEcho\Project-Echo\src\Prototypes\data\data_files\_feature_cache
Model output folder: C:\Deakin\ProjectEcho\Project-Echo\src\Prototypes\engine\torch_impl\Integrate_EfficientNetV2_Engine\_trained_models


## 4. Scan dataset

This scans all audio files and creates a table with file path and label.


In [13]:
def scan_audio_dataset(root, exts):
    rows = []
    root = pathlib.Path(root)

    if not root.exists():
        raise FileNotFoundError(f"DATA_ROOT does not exist: {root}")

    ignored_folders = {"_feature_cache", "_trained_models", "__pycache__"}

    for path in root.rglob("*"):
        if path.is_file() and path.suffix.lower() in exts:
            if any(part in ignored_folders for part in path.parts):
                continue

            label = path.parent.name

            rows.append({
                "path": str(path),
                "label": label
            })

    df = pd.DataFrame(rows)

    if df.empty:
        raise ValueError(
            "No audio files found. Check DATA_ROOT and VALID_EXTS."
        )

    return df


files_df = scan_audio_dataset(DATA_ROOT, VALID_EXTS)

print(f"Found {len(files_df)} audio files")
print(f"Found {files_df['label'].nunique()} classes")
display(files_df.head())
display(files_df["label"].value_counts().head(30))


Found 8659 audio files
Found 123 classes


,path,label
0,C:\Deakin\ProjectEcho\Project-Echo\src\Prototy...,Acanthiza chrysorrhoa
1,C:\Deakin\ProjectEcho\Project-Echo\src\Prototy...,Acanthiza chrysorrhoa
2,C:\Deakin\ProjectEcho\Project-Echo\src\Prototy...,Acanthiza chrysorrhoa
3,C:\Deakin\ProjectEcho\Project-Echo\src\Prototy...,Acanthiza chrysorrhoa
4,C:\Deakin\ProjectEcho\Project-Echo\src\Prototy...,Acanthiza chrysorrhoa


label
Rhipidura leucophrys            649
Colluricincla harmonica         610
Phylidonyris niger              543
Dasyurus maculatus              517
Rhipidura albiscapa             445
Acanthiza pusilla               249
Litoria inermis                 214
Cisticola exilis                214
Barnardius zonarius             205
Cophixalus infacetus            174
Philemon citreogularis          171
Acanthiza reguloides            160
Cincloramphus mathewsi          150
Acanthiza nana                  147
brant                           135
sheowl                          128
Petroica goodenovii             126
Haliastur sphenurus             119
Philemon corniculatus           112
spodov                          107
Vulpes vulpes                   106
Rattus Norvegicus               105
Acanthorhynchus tenuirostris    103
Melithreptus gularis            101
Rhipidura rufifrons              93
Accipiter cirrocephalus          89
Carterornis leucotis             79
jabwar                

## 5. Feature extraction functions

Each audio file is loaded, cropped or padded to 5 seconds, converted to a Mel spectrogram, and saved as `.npy` in `_feature_cache`.

This means if the notebook is restarted, already extracted features do not need to be extracted again.


In [14]:
def load_audio_fixed(path, target_sr=TARGET_SR, duration_s=DURATION_S):
    y, sr = librosa.load(path, sr=target_sr, mono=True)

    target_len = int(duration_s * target_sr)

    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)))
    else:
        y = y[:target_len]

    return y, target_sr


def melspec(y, sr):
    S = librosa.feature.melspectrogram(
        y=y,
        sr=sr,
        n_mels=N_MELS,
        hop_length=HOP_LENGTH,
        fmin=FMIN,
        fmax=FMAX
    )

    S_db = librosa.power_to_db(S, ref=np.max).astype(np.float32)
    return S_db


def feature_path_for(audio_path):
    audio_path = pathlib.Path(audio_path)

    try:
        safe_name = str(audio_path.relative_to(DATA_ROOT))
    except ValueError:
        safe_name = audio_path.name

    safe_name = safe_name.replace("\\", "_").replace("/", "_").replace(":", "")
    return CACHE_DIR / f"{safe_name}.npy"


def extract_feature_cached(audio_path):
    fp = feature_path_for(audio_path)

    if fp.exists():
        return np.load(fp)

    y, sr = load_audio_fixed(audio_path)
    feat = melspec(y, sr)

    np.save(fp, feat)
    return feat


## 6. Extract features

Run this cell to recreate all feature files. This may take time depending on the dataset size.


In [15]:
X_paths = files_df["path"].tolist()
y_labels = files_df["label"].tolist()

failed_files = []

for p in tqdm(X_paths, desc="Extracting and caching features"):
    try:
        _ = extract_feature_cached(p)
    except Exception as e:
        failed_files.append((p, str(e)))

print("Feature extraction completed.")
print("Successful files:", len(X_paths) - len(failed_files))
print("Failed files:", len(failed_files))
print("Cache folder:", CACHE_DIR)

if failed_files:
    failed_df = pd.DataFrame(failed_files, columns=["path", "error"])
    display(failed_df.head(20))


Extracting and caching features: 100%|█████████████████████████████████████████████| 8659/8659 [16:32<00:00,  8.72it/s]

Feature extraction completed.
Successful files: 8659
Failed files: 0
Cache folder: C:\Deakin\ProjectEcho\Project-Echo\src\Prototypes\data\data_files\_feature_cache


## 7. Remove failed files if any

If any files failed during feature extraction, this cell removes them from training.


In [16]:
if failed_files:
    failed_paths = {p for p, _ in failed_files}
    files_df = files_df[~files_df["path"].isin(failed_paths)].reset_index(drop=True)

X_paths = files_df["path"].tolist()
y_labels = files_df["label"].tolist()

print("Files remaining for training:", len(X_paths))
print("Classes remaining:", files_df["label"].nunique())


Files remaining for training: 8659
Classes remaining: 123


## 8. Encode labels and split train/test data

The class mapping saved later must match this label encoder.


In [24]:
le = LabelEncoder()
y = le.fit_transform(y_labels)

class_counts = pd.Series(y).value_counts()
min_class_count = class_counts.min()

if min_class_count < 2:
    raise ValueError(
        "At least one class has fewer than 2 files. "
        "Stratified train/test split requires at least 2 files per class."
    )

X_train, X_test, y_train, y_test = train_test_split(
    X_paths,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Train files:", len(X_train))
print("Test files:", len(X_test))
print("Number of classes:", len(le.classes_))
print("Classes:")
for i, label in enumerate(le.classes_):
    print(i, "->", label)


Train files: 6927
Test files: 1732
Number of classes: 123
Classes:
0 -> Acanthiza chrysorrhoa
1 -> Acanthiza lineata
2 -> Acanthiza nana
3 -> Acanthiza pusilla
4 -> Acanthiza reguloides
5 -> Acanthiza uropygialis
6 -> Acanthorhynchus tenuirostris
7 -> Accipiter cirrocephalus
8 -> Aidemosyne modesta
9 -> Alauda arvensis
10 -> Anhinga novaehollandiae
11 -> Anthochaera phrygia
12 -> Antigone rubicunda
13 -> Artamus cinereus
14 -> Artamus cyanopterus
15 -> Artamus minor
16 -> Artamus superciliosus
17 -> Barnardius zonarius
18 -> Callocephalon fimbriatum
19 -> Calyptorhynchus banksii
20 -> Calyptorhynchus lathami
21 -> Capra Hircus
22 -> Carduelis carduelis
23 -> Carterornis leucotis
24 -> Cervus Unicolour
25 -> Ceyx azureus
26 -> Chenonetta jubata
27 -> Chlamydera nuchalis
28 -> Cincloramphus mathewsi
29 -> Cinclosoma punctatum
30 -> Cisticola exilis
31 -> Climacteris picumnus
32 -> Colluricincla harmonica
33 -> Conopophila albogularis
34 -> Cophixalus exiguus
35 -> Cophixalus infacetus
36

## 9. PyTorch dataset

EfficientNetV2 receives input as `[batch, channels, n_mels, time]`.

Here, channels = 1 because the Mel spectrogram is single-channel.


In [25]:
class MelDataset(Dataset):
    def __init__(self, paths, labels):
        self.paths = list(paths)
        self.labels = np.array(labels)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        mel = extract_feature_cached(self.paths[idx])

        # Normalise each sample
        mel = (mel - np.mean(mel)) / (np.std(mel) + 1e-6)

        # Shape: [1, n_mels, time]
        mel = torch.tensor(mel, dtype=torch.float32).unsqueeze(0)

        label = torch.tensor(self.labels[idx], dtype=torch.long)

        return mel, label


## 10. Train EfficientNetV2

This uses the EfficientNetV2 model selected from Sprint 1 benchmark work.

Start with 3 epochs first to check the full pipeline. After confirming it works, increase `EPOCHS`.


In [26]:
def train_efficientnetv2(X_train, y_train, epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LEARNING_RATE):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Using device:", device)

    num_classes = len(np.unique(y_train))

    model = timm.create_model(
        "efficientnetv2_rw_s",
        pretrained=True,
        num_classes=num_classes,
        in_chans=1
    )

    model = model.to(device)

    train_ds = MelDataset(X_train, y_train)
    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0
    )

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

    history = []

    for epoch in range(epochs):
        model.train()

        running_loss = 0.0
        correct = 0
        total = 0

        for x, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
            x = x.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(x)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            running_loss += loss.item() * x.size(0)

            preds = torch.argmax(outputs, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        epoch_loss = running_loss / total
        epoch_acc = correct / total

        history.append({
            "epoch": epoch + 1,
            "train_loss": float(epoch_loss),
            "train_accuracy": float(epoch_acc)
        })

        print(f"Epoch {epoch+1}/{epochs} | Loss: {epoch_loss:.4f} | Train Acc: {epoch_acc:.4f}")

    return model, device, history


In [27]:
efficientnet_model, device, training_history = train_efficientnetv2(
    X_train,
    y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LEARNING_RATE
)


Using device: cpu


Epoch 1/3: 100%|█████████████████████████████████████████████████████████████████████| 433/433 [31:39<00:00,  4.39s/it]


Epoch 1/3 | Loss: 2.6794 | Train Acc: 0.4223


Epoch 2/3: 100%|█████████████████████████████████████████████████████████████████████| 433/433 [30:06<00:00,  4.17s/it]


Epoch 2/3 | Loss: 0.8291 | Train Acc: 0.8050


Epoch 3/3: 100%|█████████████████████████████████████████████████████████████████████| 433/433 [30:04<00:00,  4.17s/it]

Epoch 3/3 | Loss: 0.2501 | Train Acc: 0.9488


## 11. Evaluate on test set

This checks whether the trained model can produce labels and confidence values correctly.


In [28]:
@torch.no_grad()
def predict_model(model, paths, batch_size=BATCH_SIZE):
    model.eval()

    dummy_labels = np.zeros(len(paths), dtype=int)
    ds = MelDataset(paths, dummy_labels)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=0)

    all_probs = []

    for x, _ in tqdm(loader, desc="Predicting"):
        x = x.to(device)

        outputs = model(x)
        probs = torch.softmax(outputs, dim=1)

        all_probs.append(probs.cpu().numpy())

    probs = np.concatenate(all_probs, axis=0)
    preds = np.argmax(probs, axis=1)

    return preds, probs


In [29]:
y_pred, y_probs = predict_model(efficientnet_model, X_test)

acc = accuracy_score(y_test, y_pred)
f1_macro = f1_score(y_test, y_pred, average="macro")

print("Test Accuracy:", acc)
print("F1 Macro:", f1_macro)
print()
print(classification_report(y_test, y_pred, target_names=le.classes_, zero_division=0))


Predicting: 100%|████████████████████████████████████████████████████████████████████| 109/109 [02:26<00:00,  1.34s/it]

Test Accuracy: 0.8724018475750578
F1 Macro: 0.843222466600867

                              precision    recall  f1-score   support

       Acanthiza chrysorrhoa       0.80      0.57      0.67         7
           Acanthiza lineata       0.75      1.00      0.86         6
              Acanthiza nana       0.90      0.97      0.93        29
           Acanthiza pusilla       0.90      0.70      0.79        50
        Acanthiza reguloides       0.76      0.91      0.83        32
       Acanthiza uropygialis       0.83      0.77      0.80        13
Acanthorhynchus tenuirostris       0.64      0.76      0.70        21
     Accipiter cirrocephalus       0.74      0.94      0.83        18
          Aidemosyne modesta       1.00      1.00      1.00         3
             Alauda arvensis       1.00      0.80      0.89         5
     Anhinga novaehollandiae       0.00      0.00      0.00         2
         Anthochaera phrygia       0.86      0.86      0.86         7
          Antigone rubicun

## 12. Show prediction examples

It shows predicted label and confidence.


In [30]:
example_rows = []

for i in range(min(10, len(X_test))):
    true_idx = int(y_test[i])
    pred_idx = int(y_pred[i])
    confidence = float(np.max(y_probs[i]))

    example_rows.append({
        "audio_path": X_test[i],
        "true_label": le.classes_[true_idx],
        "predicted_label": le.classes_[pred_idx],
        "confidence": confidence,
        "correct": true_idx == pred_idx
    })

examples_df = pd.DataFrame(example_rows)
display(examples_df)


,audio_path,true_label,predicted_label,confidence,correct
0,C:\Deakin\ProjectEcho\Project-Echo\src\Prototy...,Barnardius zonarius,Barnardius zonarius,0.995623,True
1,C:\Deakin\ProjectEcho\Project-Echo\src\Prototy...,Rhipidura albiscapa,Rhipidura albiscapa,0.981187,True
2,C:\Deakin\ProjectEcho\Project-Echo\src\Prototy...,Colluricincla harmonica,Colluricincla harmonica,0.989218,True
3,C:\Deakin\ProjectEcho\Project-Echo\src\Prototy...,Petroica goodenovii,Petroica goodenovii,0.976628,True
4,C:\Deakin\ProjectEcho\Project-Echo\src\Prototy...,Falco peregrinus,Falco peregrinus,0.954945,True
5,C:\Deakin\ProjectEcho\Project-Echo\src\Prototy...,Cophixalus infacetus,Cophixalus infacetus,0.996565,True
6,C:\Deakin\ProjectEcho\Project-Echo\src\Prototy...,Sus Scrofa,Sus Scrofa,0.549493,True
7,C:\Deakin\ProjectEcho\Project-Echo\src\Prototy...,Petroica phoenicea,Petroica phoenicea,0.957906,True
8,C:\Deakin\ProjectEcho\Project-Echo\src\Prototy...,Phylidonyris niger,Phylidonyris niger,0.987102,True
9,C:\Deakin\ProjectEcho\Project-Echo\src\Prototy...,Colluricincla harmonica,Colluricincla harmonica,0.997531,True


## 13. Save model, class mapping, preprocessing config, and metrics

These files are needed by the Engine pipeline.


In [31]:
model_path = MODEL_DIR / "efficientnetv2_project_echo.pt"
label_map_path = MODEL_DIR / "class_mapping.json"
preprocess_path = MODEL_DIR / "preprocess_config.json"
metrics_path = MODEL_DIR / "training_metrics.json"

# Save model checkpoint
torch.save({
    "model_state_dict": efficientnet_model.state_dict(),
    "model_name": "efficientnetv2_rw_s",
    "num_classes": len(le.classes_),
    "in_chans": 1,
    "target_sr": TARGET_SR,
    "duration_s": DURATION_S,
    "n_mels": N_MELS,
    "hop_length": HOP_LENGTH,
    "fmin": FMIN,
    "fmax": FMAX
}, model_path)

# Save class mapping
class_mapping = {
    "classes": le.classes_.tolist(),
    "label_to_index": {label: int(i) for i, label in enumerate(le.classes_)},
    "index_to_label": {str(i): label for i, label in enumerate(le.classes_)}
}

with open(label_map_path, "w", encoding="utf-8") as f:
    json.dump(class_mapping, f, indent=4)

# Save preprocessing configuration
preprocess_config = {
    "target_sr": TARGET_SR,
    "duration_s": DURATION_S,
    "n_mels": N_MELS,
    "hop_length": HOP_LENGTH,
    "fmin": FMIN,
    "fmax": FMAX,
    "feature_type": "mel_spectrogram_db",
    "normalisation": "per_sample_standardisation",
    "input_shape": [1, N_MELS, "time"]
}

with open(preprocess_path, "w", encoding="utf-8") as f:
    json.dump(preprocess_config, f, indent=4)

# Save training metrics
training_metrics = {
    "model": "EfficientNetV2",
    "model_name": "efficientnetv2_rw_s",
    "test_accuracy": float(acc),
    "f1_macro": float(f1_macro),
    "train_size": len(X_train),
    "test_size": len(X_test),
    "num_classes": len(le.classes_),
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "training_history": training_history
}

with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(training_metrics, f, indent=4)

print("Saved model to:", model_path)
print("Saved class mapping to:", label_map_path)
print("Saved preprocessing config to:", preprocess_path)
print("Saved metrics to:", metrics_path)


Saved model to: C:\Deakin\ProjectEcho\Project-Echo\src\Prototypes\engine\torch_impl\Integrate_EfficientNetV2_Engine\_trained_models\efficientnetv2_project_echo.pt
Saved class mapping to: C:\Deakin\ProjectEcho\Project-Echo\src\Prototypes\engine\torch_impl\Integrate_EfficientNetV2_Engine\_trained_models\class_mapping.json
Saved preprocessing config to: C:\Deakin\ProjectEcho\Project-Echo\src\Prototypes\engine\torch_impl\Integrate_EfficientNetV2_Engine\_trained_models\preprocess_config.json
Saved metrics to: C:\Deakin\ProjectEcho\Project-Echo\src\Prototypes\engine\torch_impl\Integrate_EfficientNetV2_Engine\_trained_models\training_metrics.json


## 14. Test loading the saved model

This confirms the `.pt` file can be loaded again. This is important before connecting it to the Engine pipeline.


In [32]:
def load_saved_efficientnetv2(model_path, device="cpu"):
    checkpoint = torch.load(model_path, map_location=device)

    model = timm.create_model(
        checkpoint["model_name"],
        pretrained=False,
        num_classes=checkpoint["num_classes"],
        in_chans=checkpoint["in_chans"]
    )

    model.load_state_dict(checkpoint["model_state_dict"])
    model = model.to(device)
    model.eval()

    return model, checkpoint


In [33]:
loaded_model, checkpoint = load_saved_efficientnetv2(model_path, device=device)

sample_path = X_test[0]
sample_pred, sample_probs = predict_model(loaded_model, [sample_path])

pred_index = int(sample_pred[0])
pred_label = le.classes_[pred_index]
confidence = float(np.max(sample_probs[0]))

print("Loaded model works.")
print("Sample audio:", sample_path)
print("Predicted label:", pred_label)
print("Confidence:", confidence)


Predicting: 100%|████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  9.05it/s]

Loaded model works.
Sample audio: C:\Deakin\ProjectEcho\Project-Echo\src\Prototypes\data\data_files\Barnardius zonarius\region_64.600-66.600.mp3
Predicted label: Barnardius zonarius
Confidence: 0.9956234097480774


## 15. Final output check

After running the notebook, check that these files exist:

```text
_trained_models/
    efficientnetv2_project_echo.pt
    class_mapping.json
    preprocess_config.json
    training_metrics.json
```

**Load the selected Sprint 1 model into the real Engine pipeline.**


In [34]:
for file_path in [model_path, label_map_path, preprocess_path, metrics_path]:
    print(file_path.name, "exists:", pathlib.Path(file_path).exists())


efficientnetv2_project_echo.pt exists: True
class_mapping.json exists: True
preprocess_config.json exists: True
training_metrics.json exists: True
